In [67]:
df_orders = spark.sql("SELECT * FROM lh_urbannest.dbo.Bronze_Orders")
df_customers = spark.sql("SELECT * FROM lh_urbannest.dbo.bronze_customers")
df_products = spark.sql("SELECT * FROM lh_urbannest.dbo.Bronze_Products")

print("Orders:", df_orders.count())
print("Customers:", df_customers.count())
print("Products:", df_products.count())

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 72, Finished, Available, Finished, False)

Orders: 21721
Customers: 4200
Products: 120


In [68]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

before_count = df_orders.count()

# Rank rows within each (order_id, is_refund) group by ingestion time, descending
# Partitioning by BOTH columns preserves legitimate sale+refund pairs that share an order_id
window_spec = Window.partitionBy("order_id", "is_refund").orderBy(F.col("_ingested_at").desc())

df_orders_deduped = (
    df_orders
    .withColumn("_row_rank", F.row_number().over(window_spec))
    .filter(F.col("_row_rank") == 1)
    .drop("_row_rank")
)

after_count = df_orders_deduped.count()
print(f"Orders before dedup: {before_count}")
print(f"Orders after dedup: {after_count}")
print(f"Duplicates removed: {before_count - after_count}")

df_orders_deduped.groupBy("is_refund").count().show()

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 73, Finished, Available, Finished, False)

Orders before dedup: 21721
Orders after dedup: 21191
Duplicates removed: 530
+---------+-----+
|is_refund|count|
+---------+-----+
|        0|20570|
|        1|  621|
+---------+-----+



In [69]:
df_orders_clean = df_orders_deduped.withColumn(
    "discount_code",
    F.upper(F.trim(F.col("discount_code")))
)

# Sanity check: how many distinct discount codes before vs after
print("Distinct discount codes (raw):", df_orders_deduped.select("discount_code").distinct().count())
print("Distinct discount codes (cleaned):", df_orders_clean.select("discount_code").distinct().count())

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 74, Finished, Available, Finished, False)

Distinct discount codes (raw): 4
Distinct discount codes (cleaned): 2


In [70]:
null_channel_before = df_orders_clean.filter(F.col("channel").isNull()).count()

df_orders_clean = df_orders_clean.withColumn(
    "channel",
    F.when(F.col("channel").isNull(), F.lit("unknown")).otherwise(F.col("channel"))
)

null_channel_after = df_orders_clean.filter(F.col("channel").isNull()).count()
print(f"Null channels before: {null_channel_before}, after: {null_channel_after}")

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 75, Finished, Available, Finished, False)

Null channels before: 335, after: 0


In [71]:
df_orders_clean = df_orders_clean.withColumn(
    "order_ts_utc",
    F.to_timestamp(F.col("order_ts_utc"))
)

df_orders_clean.printSchema()

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 76, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts_utc: timestamp (nullable = true)
 |-- sku: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_code: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- device: string (nullable = true)
 |-- is_refund: string (nullable = true)
 |-- _ingested_at: string (nullable = true)
 |-- _source_file: string (nullable = true)



In [72]:
from datetime import datetime

dq_log = [
    {
        "check_name": "duplicate_order_id_removal",
        "table": "orders",
        "before_count": before_count,
        "after_count": after_count,
        "rows_affected": before_count - after_count,
        "run_timestamp": str(datetime.utcnow())
    },
    {
        "check_name": "null_channel_handling",
        "table": "orders",
        "before_count": null_channel_before,
        "after_count": null_channel_after,
        "rows_affected": null_channel_before,
        "run_timestamp": str(datetime.utcnow())
    }
]

df_dq_log = spark.createDataFrame(dq_log)
display(df_dq_log)

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 77, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2b32f2d2-afcb-4b14-8da4-b4bc3ddde6f6)

In [73]:
df_orders_clean = (
    df_orders_clean
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("discount_amount", F.col("discount_amount").cast("double"))
    .withColumn("revenue", F.col("revenue").cast("double"))
    .withColumn("is_refund", F.col("is_refund").cast("int"))
    .withColumn("_ingested_at", F.to_timestamp(F.col("_ingested_at")))
)

df_orders_clean.printSchema()

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 78, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts_utc: timestamp (nullable = true)
 |-- sku: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_code: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- device: string (nullable = true)
 |-- is_refund: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [74]:
print("=== Customers ===")
df_customers.printSchema()
print("Total rows:", df_customers.count())
print("Distinct customer_id:", df_customers.select("customer_id").distinct().count())
print("Null signup_date:", df_customers.filter(F.col("signup_date").isNull()).count())
print("Null region:", df_customers.filter(F.col("region").isNull()).count())

print("\n=== Products ===")
df_products.printSchema()
print("Total rows:", df_products.count())
print("Distinct sku:", df_products.select("sku").distinct().count())
print("Null list_price:", df_products.filter(F.col("list_price").isNull()).count())

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 79, Finished, Available, Finished, False)

=== Customers ===
root
 |-- customer_id: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- _ingested_at: string (nullable = true)
 |-- _source_file: string (nullable = true)

Total rows: 4200
Distinct customer_id: 4200
Null signup_date: 0
Null region: 0

=== Products ===
root
 |-- sku: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- list_price: string (nullable = true)
 |-- _ingested_at: string (nullable = true)
 |-- _source_file: string (nullable = true)

Total rows: 120
Distinct sku: 120
Null list_price: 0


In [75]:
df_customers_clean = (
    df_customers
    .withColumn("signup_date", F.to_date(F.col("signup_date")))
    .withColumn("_ingested_at", F.to_timestamp(F.col("_ingested_at")))
)

df_products_clean = (
    df_products
    .withColumn("list_price", F.col("list_price").cast("double"))
    .withColumn("_ingested_at", F.to_timestamp(F.col("_ingested_at")))
)

df_customers_clean.printSchema()
df_products_clean.printSchema()

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 80, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

root
 |-- sku: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- list_price: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [76]:
df_orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")
df_customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")
df_products_clean.write.format("delta").mode("overwrite").saveAsTable("silver_products")

print("Silver tables written successfully.")

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 81, Finished, Available, Finished, False)

Silver tables written successfully.


In [77]:
print("silver_orders:", spark.sql("SELECT COUNT(*) FROM silver_orders").collect()[0][0])
print("silver_customers:", spark.sql("SELECT COUNT(*) FROM silver_customers").collect()[0][0])
print("silver_products:", spark.sql("SELECT COUNT(*) FROM silver_products").collect()[0][0])

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 82, Finished, Available, Finished, False)

silver_orders: 21191
silver_customers: 4200
silver_products: 120


In [78]:
df_orders_clean.select("quantity", "unit_price", "revenue", "discount_amount").describe().show()

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 83, Finished, Available, Finished, False)

+-------+------------------+------------------+------------------+------------------+
|summary|          quantity|        unit_price|           revenue|   discount_amount|
+-------+------------------+------------------+------------------+------------------+
|  count|             21191|             21191|             21191|             21191|
|   mean| 1.155301779057147|3748.1139710254515| 4111.227055825612|229.86853381152386|
| stddev|0.6677320282803959|1734.3360308752842|3281.8914794473963| 664.0573169431653|
|    min|                -3|            587.72|         -19832.34|          -4715.96|
|    max|                 3|           6788.86|          20186.07|           5966.34|
+-------+------------------+------------------+------------------+------------------+



In [80]:
df_orders_clean.filter(F.col("is_refund") == 1).select("revenue").describe().show()
df_orders_clean.filter(F.col("is_refund") == 1).count()

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 85, Finished, Available, Finished, False)

+-------+------------------+
|summary|           revenue|
+-------+------------------+
|  count|               621|
|   mean|-4182.942222222224|
| stddev|  2835.19093905608|
|    min|         -19832.34|
|    max|           -418.19|
+-------+------------------+



621

In [88]:
from pyspark.sql import functions as F

# Get the true min/max date range from your actual order data
date_range = df_orders_clean.select(
    F.min("order_ts_utc").alias("min_date"),
    F.max("order_ts_utc").alias("max_date")
).collect()[0]

print("Date range:", date_range["min_date"], "to", date_range["max_date"])

# Generate one row per calendar date across that range
df_dim_date = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{date_range['min_date']}'),
        to_date('{date_range['max_date']}'),
        interval 1 day
    )) AS date
""")

df_dim_date = (
    df_dim_date
    .withColumn("date_id", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin([1, 7]))
)

display(df_dim_date.limit(10))

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 93, Finished, Available, Finished, False)

Date range: 2025-12-01 00:34:49 to 2026-10-31 23:59:15


SynapseWidget(Synapse.DataFrame, 4fd2a6d0-6e69-4116-b8a8-ea30551a3426)

In [89]:
df_dim_customer = df_customers_clean.select(
    F.col("customer_id"),
    F.col("region"),
    F.col("signup_date")
)

print("dim_customer rows:", df_dim_customer.count())
display(df_dim_customer.limit(5))

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 94, Finished, Available, Finished, False)

dim_customer rows: 4200


SynapseWidget(Synapse.DataFrame, 954f8ddc-016f-4ba7-ac0b-5ecb1c645341)

In [90]:
df_dim_product = df_products_clean.select(
    F.col("sku"),
    F.col("product_name"),
    F.col("category"),
    F.col("list_price")
)

print("dim_product rows:", df_dim_product.count())
display(df_dim_product.limit(5))

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 95, Finished, Available, Finished, False)

dim_product rows: 120


SynapseWidget(Synapse.DataFrame, 8897e150-9082-4272-9026-c0b5715f5d81)

In [91]:
df_fact_orders = (
    df_orders_clean
    .withColumn("date_id", F.date_format("order_ts_utc", "yyyyMMdd").cast("int"))
    .select(
        F.col("order_id"),
        F.col("customer_id"),
        F.col("sku"),
        F.col("date_id"),
        F.col("order_ts_utc"),
        F.col("quantity"),
        F.col("unit_price"),
        F.col("discount_code"),
        F.col("discount_amount"),
        F.col("revenue"),
        F.col("channel"),
        F.col("device"),
        F.col("is_refund")
    )
)

print("fact_orders rows:", df_fact_orders.count())
display(df_fact_orders.limit(5))

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 96, Finished, Available, Finished, False)

fact_orders rows: 21191


SynapseWidget(Synapse.DataFrame, 16323d7a-a622-4189-8c49-14f629f63dc9)

In [94]:
# Orphan check: customer_ids in fact_orders with no match in dim_customer
orphan_customers = df_fact_orders.join(df_dim_customer, "customer_id", "left_anti").count()
print("Orphan customer_ids (no match in dim_customer):", orphan_customers)

# Orphan check: skus in fact_orders with no match in dim_product
orphan_products = df_fact_orders.join(df_dim_product, "sku", "left_anti").count()
print("Orphan skus (no match in dim_product):", orphan_products)

# Orphan check: date_ids in fact_orders with no match in dim_date
orphan_dates = df_fact_orders.join(df_dim_date, "date_id", "left_anti").count()
print("Orphan date_ids (no match in dim_date):", orphan_dates)

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 100, Finished, Available, Finished, False)

Orphan customer_ids (no match in dim_customer): 0
Orphan skus (no match in dim_product): 0
Orphan date_ids (no match in dim_date): 0


In [95]:
df_fact_orders.write.format("delta").mode("overwrite").saveAsTable("gold_fact_orders")
df_dim_customer.write.format("delta").mode("overwrite").saveAsTable("gold_dim_customer")
df_dim_product.write.format("delta").mode("overwrite").saveAsTable("gold_dim_product")
df_dim_date.write.format("delta").mode("overwrite").saveAsTable("gold_dim_date")

print("Gold tables written successfully.")

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 101, Finished, Available, Finished, False)

Gold tables written successfully.


In [96]:
print("gold_fact_orders:", spark.sql("SELECT COUNT(*) FROM gold_fact_orders").collect()[0][0])
print("gold_dim_customer:", spark.sql("SELECT COUNT(*) FROM gold_dim_customer").collect()[0][0])
print("gold_dim_product:", spark.sql("SELECT COUNT(*) FROM gold_dim_product").collect()[0][0])
print("gold_dim_date:", spark.sql("SELECT COUNT(*) FROM gold_dim_date").collect()[0][0])

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 102, Finished, Available, Finished, False)

gold_fact_orders: 21191
gold_dim_customer: 4200
gold_dim_product: 120
gold_dim_date: 335


In [97]:
spark.sql("""
    SELECT 
        d.year, d.month, d.month_name,
        SUM(CASE WHEN f.is_refund = 0 THEN f.revenue ELSE 0 END) AS gross_sales_revenue,
        SUM(CASE WHEN f.is_refund = 1 THEN f.revenue ELSE 0 END) AS refund_revenue,
        SUM(f.revenue) AS net_revenue,
        COUNT(CASE WHEN f.is_refund = 0 THEN 1 END) AS order_count,
        SUM(CASE WHEN f.discount_code IS NOT NULL AND f.is_refund = 0 THEN 1 ELSE 0 END) AS discounted_orders,
        ROUND(100.0 * SUM(CASE WHEN f.discount_code IS NOT NULL AND f.is_refund = 0 THEN 1 ELSE 0 END) 
              / COUNT(CASE WHEN f.is_refund = 0 THEN 1 END), 1) AS pct_orders_discounted
    FROM gold_fact_orders f
    JOIN gold_dim_date d ON f.date_id = d.date_id
    GROUP BY d.year, d.month, d.month_name
    ORDER BY d.year, d.month
""").show(20, truncate=False)

StatementMeta(, 8f73686c-02f4-4c69-9e4f-a506ed9b43fc, 103, Finished, Available, Finished, False)

+----+-----+----------+-------------------+-------------------+------------------+-----------+-----------------+---------------------+
|year|month|month_name|gross_sales_revenue|refund_revenue     |net_revenue       |order_count|discounted_orders|pct_orders_discounted|
+----+-----+----------+-------------------+-------------------+------------------+-----------+-----------------+---------------------+
|2025|12   |December  |8491711.980000013  |-292944.36999999994|8198767.610000012 |1900       |157              |8.3                  |
|2026|1    |January   |7985266.07999999   |-238313.63999999998|7746952.439999988 |1750       |145              |8.3                  |
|2026|2    |February  |7577986.119999983  |-223282.15         |7354703.969999983 |1700       |129              |7.6                  |
|2026|3    |March     |8200488.180000005  |-208714.28         |7991773.900000005 |1820       |111              |6.1                  |
|2026|4    |April     |8045393.979999997  |-172523.0299